# 04.2 · q-transfer FAILURE analysis — single examples

Near-identical to `04_q_transfer.ipynb`, but instead of generating fresh setups it **reconstructs instances that FAILED causally in the completed run** (`results/diff/{q_transfer,filter_transfer}`) and re-runs the same intervention + eval on them — one at a time, for debugging/finetuning. Prompts go through the current (fixed) `materialize` so the pivot is a real token; gaze uses the new absolute `gaze(SRC) > gaze(DST)` metric with the clean baseline.

_A scaled-up version comes later; for now these are single hand-picked failures._

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4, 7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-03 13:24:57 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Load failure cases from the completed run

`rebuild()` reconstructs the data instance from a saved record (q from `src_progress + [expected_next]`, filter from the record key). `load_fails` keeps only causal failures; we prioritize **readout-shifted** failures (`readout.patched == src` but the generation didn't follow) — the most diagnostic 'the model knew but didn't act' cases.

In [2]:
import json, glob
from pathlib import Path
from src.utils import env_utils

MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"
Q_ROOT = Path(env_utils.DEFAULT_RESULTS_DIR) / "diff" / "q_transfer" / MODEL_SHORT
F_ROOT = Path(env_utils.DEFAULT_RESULTS_DIR) / "diff" / "filter_transfer" / MODEL_SHORT

def rebuild(r):
    """Reconstruct the data instance object from one saved record."""
    dt = r["datatype"]
    if r.get("setup", "q") == "filter":
        ft = D.FILTER_TASKS[dt]
        kv = dict(p.split("=", 1) for p in r["key"].split("|") if "=" in p)
        return D.FilterTransferInstance(
            task=ft, pair=tuple(kv["pair"].split("-")), var_name=ft.var_names[0],
            src_lang=r["src_lang"], dst_lang=r["dst_lang"], src_fill=kv["srcfill"], dst_fill=kv["dstfill"],
            expected_next=kv["dstfill"], postfix=r["naming"], meta={"datatype": dt, "dst_next": kv["srcfill"]})
    bank = D.QTASKS[dt]; by = {p.id: p for p in bank.predicates}
    ids = list(r["src_progress"]) + [r["expected_next"]]
    task = D.dataclasses.replace(bank, id=f"{bank.id}_k{len(ids)}", predicates=tuple(by[i] for i in ids))
    return D.QTransferInstance(
        task=task, var_name=bank.var_names[0], presentation=r["presentation"], src_lang=r["src_lang"],
        dst_lang=r["dst_lang"], src_progress=list(r["src_progress"]), dst_progress=list(r["dst_progress"]),
        expected_next=r["expected_next"], naming=r["naming"] if r["naming"] in ("natural", "letters") else "natural",
        meta={"datatype": dt, "dst_next": r["dst_next"]})

def load_fails(root):
    out = []
    for f in glob.glob(f"{root}/*/results.json"):
        for r in json.loads(Path(f).read_text()):
            if r.get("causal") and not r["causal"]["triggered"]:
                r["_cfg"] = Path(f).parent.name; out.append(r)
    return out

def show_fails(fails, k=12):
    ro_shift = [r for r in fails if (r.get("readout") or {}).get("patched") in ("src", "both")]
    print(f"{len(fails)} causal failures  ({len(ro_shift)} readout-shifted — prioritized)")
    for r in ro_shift[:k]:
        ro = r.get("readout") or {}
        print(f"  [{r['_cfg']}] SRC {r['expected_next']!r} -> DST {r['dst_next']!r} | "
              f"causal={r['causal']['label']} · readout_patched={ro.get('patched')}")
    return ro_shift or fails

def pick_failure(fails, i=0):
    """Deterministic pick — readout-shifted failures first, then the i-th."""
    ro_shift = [r for r in fails if (r.get("readout") or {}).get("patched") in ("src", "both")]
    pool = ro_shift or fails
    return pool[i % len(pool)]

qfails = load_fails(Q_ROOT)
qcands = show_fails(qfails)

172 causal failures  (141 readout-shifted — prioritized)
  [q__int__randnm__python-cpp] SRC 'absolute_value' -> DST 'digit_count' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'last_digit' -> DST 'is_positive' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'absolute_value' -> DST 'is_positive' | causal=dst · readout_patched=both
  [q__int__randnm__python-cpp] SRC 'is_prime' -> DST 'is_negative' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'is_perfect_square' -> DST 'is_palindrome' | causal=neither · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'sign' -> DST 'is_negative' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'sign' -> DST 'is_positive' | causal=dst · readout_patched=both
  [q__int__randnm__python-cpp] SRC 'is_positive' -> DST 'is_even' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'is_multiple_of_3' -> DST 'is_negative' | causal=dst · readout_patched

## Render one q-transfer failure — SRC and DST prompts with pivots

In [3]:
qpick = pick_failure(qfails, i=0)          # <- change i to inspect a different failure
inst = rebuild(qpick)
m = D.materialize(inst)                     # current (fixed) chat template
src_prompt, dst_prompt = m["src_prompt"], m["dst_prompt"]
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
print(f"[{qpick['_cfg']}]  expected_next = {inst.expected_next!r}  ({inst.expected_next_desc})")
print(f"OLD run verdict: causal={qpick['causal']['label']} (FAILED) · "
      f"readout {qpick.get('readout', {}).get('source')}->{qpick.get('readout', {}).get('dest')}->"
      f"{qpick.get('readout', {}).get('patched')}")
show(src_prompt, f"SRC · progress={inst.src_progress} · about to write {inst.expected_next!r} · pivot@{piv_src}", piv=-1)
show(dst_prompt, f"DST · progress={inst.dst_progress} · pivot@{piv_dst}", piv=-1)

[q__int__randnm__python-cpp]  expected_next = 'absolute_value'  (the absolute value)
OLD run verdict: causal=dst (FAILED) · readout src->dst->src


╭─ SRC · progress=['is_even', 'digit_count'] · about to write 'absolute_value' · pivot@121 ───────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given an integer `n`, computes:\n               │
│ (1) whether the number is even\n                                                                                │
│ (2) the number of digits\n                                                                                      │
│ (3) the absolute value\n                                                                                        │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(n: int):\n                                                                                             │
│     is_even = (n % 2 == 0)\n                                                                                    │
│     digit_count = len(str(abs(n)))\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · progress=['is_even'] · pivot@124 ────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given an integer `n`, computes:\n                  │
│ (1) whether the number is even\n                                                                                │
│ (2) the number of digits\n                                                                                      │
│ (3) the absolute value\n                                                                                        │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(int n){\n                                                                                             │
│     bool is_even = (n % 2 == 0);\n                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant the SRC pivot query into DST

Cache the (pre-RoPE) query of `PATCH_HEADS` at the SRC pivot; inject it at the DST pivot (prefill + re-clamped for `n_steer` decode steps). Expect the patched DST to implement `expected_next` instead of its own next predicate.

In [4]:
# ── intervention: transplant the SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_src = cache_src_proj(src_prompt, [piv_src], Q_LAYERS, "q")

def q_inject_specs(qcache, dest_pivot, heads=PATCH_HEADS):
    qh = heads_per_layer(heads, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],   # prefill
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])            # decode re-clamp

src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)
pf, dc  = q_inject_specs(q_src, piv_dst)

# clean baseline attn over the TASK (numbered predicate list); query = pivot (last token)
_spans    = [token_span(dst_prompt, p.description) for p in inst.task.predicates]
TASK_POS  = list(range(min(s[0] for s in _spans), max(s[1] for s in _spans)))
task_toks = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(dst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, piv_dst)

# ── source_run — clean SRC (frontier = expected_next); generation + goalscope ──
gen_src = greedy(src_tok, max_new=96)
rich_print(f"[red1]source_run · clean SRC (frontier {inst.expected_next!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_prompt, piv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); generation + goalscope + ATTN_VIS baseline ──
gen_dst = greedy(dst_tok, max_new=96)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.meta['dst_next']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_prompt, piv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
attn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); gen + goalscope + ATTN_DIFF ──
_iface = make_patch_interface(mt, pf)
gen_patched = greedy(dst_tok, prefill_specs=pf, decode_specs=dc, n_steer=32, max_new=96)
rich_print(f"[salmon1]patched_run · DST (expect {inst.expected_next!r}):[/salmon1] \n{escape(gen_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(dst_tok, mt, value_weighted=True, patch_interface=_iface)
attn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, piv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, piv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, attn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'absolute_value'): 
 abs_value = abs(n)
    return is_even, digit_count, abs_value
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=121 '    ' => I'm evaluating the current value of the variable and determining its
absolute value.

destination_run · clean DST (frontier 'digit_count'): 
 int num_digits = (n == 0) ? 1 : log10(abs(n)) + 1;
    int abs_value = abs(n);
    return make_tuple(is_even, num_digits, abs_value);
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=124 '    ' => I'm evaluating the absolute value of the variable.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.017)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=1.143)

---
## Evaluate the intervention (`src.diff.eval`)

Three complementary evals of the q-transfer intervention (single source of truth in `src/diff/eval.py`):

1. **Causal** (what we care about) — oracle judges whether the patched continuation pursues the SRC target (or a non-trivial part).
2. **Readout shift** — classifies all three goalscope readouts (source/destination/patched); a case is *valid* only if the clean readouts land as expected (source→SRC, dest→DST), and *shifted* if the patched readout expresses the SRC target.
3. **Gaze** — rule-based: did the value-weighted attention delta move toward the SRC-target tokens vs the DST-target tokens?

The causal & readout evals call the Claude oracle (`ask_claude`, disk-cached). The same `eval_q_transfer` works on the composed-filter instance below.

In [5]:
from src.diff import eval as E

qeval = E.eval_q_transfer(
    inst,
    patched_gen=gen_patched,                                   # 1. causal (oracle)
    readouts=(ro_src, ro_dst, ro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=dst_prompt, task_positions=TASK_POS,     # 3. gaze (rule-based, absolute)
              attn_patched=attn_patched_goal, attn_clean=attn_clean_goal),
)
rich_print(f"[bold]EVAL[/bold] · expected_next={inst.expected_next!r}\n{qeval.summary()}")
rich_print("  causal :", qeval.causal)
rich_print("  readout:", qeval.readout)
rich_print("  gaze   :", qeval.gaze)

EVAL · expected_next='absolute_value'
causal: no (dst) | readout: src→dst→src  | gaze: src+0.741 vs dst+0.079 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'dst',
    'why': 'The first statement names the variable `num_digits`, expressing intent to compute the number of digits,
even though the formula used is actually an absolute value computation.',
    'raw': 'The first statement is `int num_digits = (n < 0) ? -n : n;` — the variable is named `num_digits` but 
the computation is actually computing the absolute value (negating if negative). However, the variable name 
suggests intent toward number of digits, while the actual computation is the absolute value formula.\n\nLooking at 
the naming: `num_digits` is assigned the absolute value computation. This is ambiguous, but the variable name 
`num_digits` suggests it\'s meant to be used for computing the number of digits (perhaps as a first step). The next
line computes `abs_value = std::abs(n)` separately.\n\nThe first statement\'s variable name is `num_digits`, 
suggesting it pursues (B), but the computation itself is an absolute value. Given the variable name explicitly says
"num_digits," I\'ll classify based on the expressed intent.\n\n<verdict>B</verdict>\n<why>The first statement names
the variable `num_digits`, expressing intent to compute the number of digits, even though the formula used is 
actually an absolute value computation.</why>',
    'triggered': False
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase explicitly mentions "determining its absolute value," which directly corresponds to 
sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly mentions "determining its absolute value," 
which directly corresponds to sub-goal (A).</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly describes counting digits by converting to string and using len(), which 
directly pursues sub-goal (B): the number of digits.',
            'raw': '<verdict>B</verdict>\n<why>The phrase explicitly describes counting digits by converting to 
string and using len(), which directly pursues sub-goal (B): the number of digits.</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly states "evaluating the absolute value," directly matching sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly states "evaluating the absolute value," 
directly matching sub-goal (A).</why>'
        }
    }
}

gaze   :
{
    'src_patched': 0.7413519670565923,
    'dst_patched': 0.07927130721509457,
    'shifted': True,
    'margin': 0.6620806598414978,
    'src_clean': 0.10303397476673126,
    'dst_clean': 0.29988817730918527,
    'clean_prefers_src': False
}

---
## Composed-filter FAILURE (`filter_transfer_setups`)

Same as the composed-filter section of `04_q_transfer.ipynb`, seeded on a causal failure from the completed **filter-transfer** run. The current `materialize` fix (pivot on `if`, not the dangling space) applies here — some of these flip to PASS on re-run.

In [11]:
ffails = load_fails(F_ROOT)
fcands = show_fails(ffails)

450 causal failures  (382 readout-shifted — prioritized)
  [filter__list_str__targeted__python-python] SRC 'palindrome' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'all_upper' -> DST 'capitalized' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'same_reversed' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'starts_a' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'short' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'all_lower' -> DST 'all_alpha' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'capitalized' -> DST 'all_alpha' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'length_five' -> DST 'all_upper' | causal=dst · readout_patched=both
  [filte

In [28]:
fpick = pick_failure(ffails, i=57)          # <- change i to inspect a different failure
finst = rebuild(fpick)
fm = D.materialize(finst)
fsrc, fdst = fm["src_prompt"], fm["dst_prompt"]
fpiv_src, fpiv_dst = pivot_of(fsrc), pivot_of(fdst)
print(f"[{fpick['_cfg']}]  expected_next = {finst.expected_next!r}  ({finst.expected_next_desc})")
print(f"OLD run verdict: causal={fpick['causal']['label']} (FAILED) · "
      f"readout_patched={fpick.get('readout', {}).get('patched')}")
show(fsrc, f"SRC · prefilled {finst.src_fill!r} · about to apply {finst.expected_next!r} · pivot@{fpiv_src}",
     emph_span=token_span(fsrc, fm["task_text"]), piv=-1)
show(fdst, f"DST · prefilled {finst.dst_fill!r} · about to apply {finst.meta['dst_next']!r} · pivot@{fpiv_dst}",
     emph_span=token_span(fdst, fm["task_text"]), piv=-1)

[filter__list_str__targeted__python-python]  expected_next = 'starts_a'  (start with the letter a)
OLD run verdict: causal=dst (FAILED) · readout_patched=both


╭─ SRC · prefilled 'capitalized' · about to apply 'starts_a' · pivot@123 ─────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that start with an uppercase letter and start with the letter a\n                      │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w[:1].isupper()]\n                                                               │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · prefilled 'starts_a' · about to apply 'capitalized' · pivot@121 ─────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that start with an uppercase letter and start with the letter a\n                      │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w.startswith('a')]\n                                                             │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Intervention — same query transplant, TASK span = the composed phrase

Reuses `q_inject_specs` / `Q_LAYERS` from the q-transfer intervention above; the only change is `TASK_POS = token_span(fdst, m["task_text"])`.

In [29]:
PATCH_HEADS = GOAL_HEADS  # heads carrying the transplant (GOAL | UNION | ...)
len(PATCH_HEADS)

122

In [30]:
# ── composed-filter intervention: transplant SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS  = sorted({l for l, _ in PATCH_HEADS})
fq_src    = cache_src_proj(fsrc, [fpiv_src], Q_LAYERS, "q")
fsrc_tok  = prepare_input(prompts=fsrc, tokenizer=mt)
fdst_tok  = prepare_input(prompts=fdst, tokenizer=mt)
fpf, fdc  = q_inject_specs(fq_src, fpiv_dst)              # SAME machinery as the q-transfer cell

# TASK span = the single COMPOSED phrase (FilterTask has .conditions, not a per-line list); query = pivot
TASK_POS  = list(range(*token_span(fdst, fm["task_text"])))
task_toks = [mt.tokenizer.decode([int(fdst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(fdst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, fpiv_dst)

# ── source_run — clean SRC (frontier = expected_next) ──
fgen_src = greedy(fsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC (frontier {finst.expected_next!r}):[/red1] \n{escape(fgen_src)}")
fro_src = goalscope_readout("   ↳ goalscope · source_run", fsrc, fpiv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); ATTN_VIS baseline ──
fgen_dst = greedy(fdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {finst.meta['dst_next']!r}):[/sea_green3] \n{escape(fgen_dst)}")
fro_dst = goalscope_readout("   ↳ goalscope · destination_run", fdst, fpiv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
fattn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); ATTN_DIFF ──
_iface = make_patch_interface(mt, fpf)
fgen_patched = greedy(fdst_tok, prefill_specs=fpf, decode_specs=fdc, n_steer=32, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {finst.expected_next!r}):[/salmon1] \n{escape(fgen_patched)}")
fro_patched = goalscope_readout("   ↳ goalscope · patched_run", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(fdst_tok, mt, value_weighted=True, patch_interface=_iface)
fattn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, fpiv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, fpiv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, fattn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'starts_a'): 
 w.startswith('A')]

# Example usage:
# words = ["Apple", "banana", "Apricot", "avocado", "Almond"]
# print(func(words))  # Output: ['Apple', 'Apricot', 'Almond']
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=123 ' if' => I'm checking if the variable starts with 'A'.

destination_run · clean DST (frontier 'capitalized'): 
 w[0].isupper()]

# Example usage:
# words = ["Apple", "banana", "Apricot", "avocado", "Almond"]
# print(func(words))  # Output: ['Apple', 'Apricot', 'Almond']
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=121 ' if' => I'm checking if the character is uppercase.

... (2 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=121 ' if' => I'm checking if the character is 'A'.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.034)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=0.440)

### Evaluate the composed-filter intervention

Same `eval_q_transfer` (the filter instance exposes the same `expected_next_desc` / `dst_next_desc` / `task_desc`), on the filter run's captured outputs.

In [27]:
feval = E.eval_q_transfer(
    finst,
    patched_gen=fgen_patched,                                    # 1. causal (oracle)
    readouts=(fro_src, fro_dst, fro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=fdst, task_positions=TASK_POS,              # 3. gaze (rule-based, absolute)
              attn_patched=fattn_patched_goal, attn_clean=fattn_clean_goal),
)
rich_print(f"[bold]EVAL (filter)[/bold] · expected_next={finst.expected_next!r}\n{feval.summary()}")
rich_print("  causal :", feval.causal)
rich_print("  readout:", feval.readout)
rich_print("  gaze   :", feval.gaze)

EVAL (filter) · expected_next='has_double'
causal: TRIGGERED (src) | readout: src→dst→src  | gaze: src+0.456 vs dst+0.065 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'src',
    'why': 'The code checks for repeated adjacent characters (w[i] == w[i+1]) — there is no palindrome check 
present in this first (and only) statement.',
    'raw': '<verdict>A</verdict>\n<why>The code checks for repeated adjacent characters (w[i] == w[i+1]) — there is
no palindrome check present in this first (and only) statement.</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase explicitly describes checking for a repeated adjacent character (same character 
immediately following another), which is exactly sub-goal A.',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly describes checking for a repeated adjacent 
character (same character immediately following another), which is exactly sub-goal A.</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'Checking if a string equals its reverse is the definition of a palindrome check, which is 
sub-goal B.',
            'raw': '<verdict>B</verdict>\n<why>Checking if a string equals its reverse is the definition of a 
palindrome check, which is sub-goal B.</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly describes checking for a repeated adjacent character (same character 
immediately following another), which is exactly sub-goal A.',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly describes checking for a repeated adjacent 
character (same character immediately following another), which is exactly sub-goal A.</why>'
        }
    }
}

gaze   :
{
    'src_patched': 0.456238878518343,
    'dst_patched': 0.06522940937429667,
    'shifted': True,
    'margin': 0.39100946914404633,
    'src_clean': 0.12277377396821976,
    'dst_clean': 0.397003710269928,
    'clean_prefers_src': False
}